In [0]:
# Packages required by all code.
# Versions of Databricks code are not locked since Databricks ensures changes are backwards compatible.
# Versions of open source packages are locked since package authors often make backwards compatible changes
%pip install -qqqq -U \
  databricks-vectorsearch databricks-agents pydantic databricks-sdk mlflow mlflow-skinny `# For agent & data pipeline code` \
  pypdf==4.1.0  `# PDF parsing` \
  markdownify==0.12.1  `# HTML parsing` \
  pypandoc_binary==1.13  `# DOCX parsing` \
  transformers==4.41.1 torch==2.3.0 tiktoken==0.7.0 langchain-text-splitters==0.2.0. `# get_recursive_character_text_splitter` \

# Restart to load the packages into the Python environment
dbutils.library.restartPython()

In [0]:
%run ./00_config

In [0]:
import mlflow

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

## Start MLflow run for tracking

In [0]:
mlflow.start_run(run_name=POC_DATA_PIPELINE_RUN_NAME)

## Prepare data for creating embeddings and the vector search index

### Chunk data



In [0]:
from typing import Literal, Optional, Any, Callable
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql.functions import explode
import pyspark.sql.functions as func
from typing import Callable
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
import tiktoken
from pyspark.sql.types import StructType, StringType, StructField, MapType, ArrayType

In [0]:
# tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-large-en-v1.5")
# context_window = 512

chunk_spec = {
  "tokenizer": lambda: AutoTokenizer.from_pretrained(
  "BAAI/bge-large-en-v1.5"),
  "chunk_size_tokens": 368,
  "chunk_overlap_tokens": 40,
  "context_window": 512,
  "type": "SENTENCE_TRANSFORMER"
}

In [0]:
def validate_chunk_size(chunk_spec: dict):
    """
    Validate the chunk size and overlap settings in chunk_spec.
    Raises ValueError if any condition is violated.
    """
    if (
        chunk_spec["chunk_overlap_tokens"] + chunk_spec["chunk_size_tokens"]
    ) > chunk_spec["context_window"]:
        raise ValueError(
            f'Proposed chunk_size of {chunk_spec["chunk_size_tokens"]} + overlap of {chunk_spec["chunk_overlap_tokens"]} '
            f'is {chunk_spec["chunk_overlap_tokens"] + chunk_spec["chunk_size_tokens"]} which is greater than context '
            f'window of {chunk_spec["context_window"]} tokens.'
        )

    if chunk_spec["chunk_overlap_tokens"] > chunk_spec["chunk_size_tokens"]:
        raise ValueError(
            f'Proposed `chunk_overlap_tokens` of {chunk_spec["chunk_overlap_tokens"]} is greater than the '
            f'`chunk_size_tokens` of {chunk_spec["chunk_size_tokens"]}. Reduce the size of `chunk_size_tokens`.'
        )


def get_recursive_character_text_splitter(
    chunk_spec: dict,
    embedding_model_name: str = EMBEDDING_MODEL_NAME,
    # chunk_overlap_tokens: int = 0,
) -> Callable[[str], list[str]]:

    # Validate chunk size and overlap
    validate_chunk_size(chunk_spec)

    print(f'Chunk size in tokens: {chunk_spec["chunk_size_tokens"]}')
    print(f'Chunk overlap in tokens: {chunk_spec["chunk_overlap_tokens"]}')
    context_usage = (
      round(
          (chunk_spec["chunk_size_tokens"] + chunk_spec["chunk_overlap_tokens"])
          / chunk_spec["context_window"],
          2,
      )
      * 100
    )
    print(
      f'Using {context_usage}% of the {chunk_spec["context_window"]} token context window.'
    )

    def _recursive_character_text_splitter(text: str) -> list[str]:
        tokenizer = chunk_spec["tokenizer"]()
        if chunk_spec["type"] == "SENTENCE_TRANSFORMER":
            splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
                tokenizer,
                chunk_size=chunk_spec["chunk_size_tokens"],
                chunk_overlap=chunk_spec["chunk_overlap_tokens"],
            )
        elif chunk_spec["type"] == "OPENAI":
            splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                tokenizer.name,
                chunk_size=chunk_spec["chunk_size_tokens"],
                chunk_overlap=chunk_spec["chunk_overlap_tokens"],
            )
        else:
            raise ValueError(f"Unsupported model type: {chunk_spec['type']}")
        try:
            return splitter.split_text(text)
        except Exception as e:
            print(e)

    return _recursive_character_text_splitter

In [0]:
def compute_chunks(
    docs_table: str,
    doc_column: str,
    chunk_fn: Callable[[str], list[str]],
    chunked_docs_table: str,
) -> str:
    chunked_docs_table = chunked_docs_table or f"{docs_table}_chunked"

    print(f"Computing chunks for `{docs_table}`...")

    raw_docs = spark.read.table(docs_table)

    # use a udf to parallelize
    parser_udf = func.udf(
        chunk_fn,
        returnType=ArrayType(StringType()),
    )
    chunked_array_docs = raw_docs.withColumn(
        "raw_content_chunked", parser_udf(doc_column)
    )#.drop(doc_column)
    chunked_docs = chunked_array_docs.select(
        "*", explode("raw_content_chunked").alias("content_chunked")
    )

    # Add a primary key: "chunk_id".
    chunks_with_ids = chunked_docs.withColumn(
        "chunk_id", func.md5(func.col("content_chunked"))
    )

    # Reorder for better display.
    chunks_with_ids = chunks_with_ids.select(
        "chunk_id", "content_chunked", *raw_docs.columns
    )

    print(f"Created {chunks_with_ids.count()} chunks!")

    # Write to Delta Table
    chunks_with_ids.write.mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(chunked_docs_table)

    return chunked_docs_table

In [0]:
TABLE_NAME = "wikipedia_movies_raw"
INPUT_DELTA_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.{TABLE_NAME}"
CHUNKED_DELTA_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.wikipedia_chunked"
VECTOR_INDEX_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.{TABLE_NAME}_vector_index"


In [0]:
chunk_fn = get_recursive_character_text_splitter(
    embedding_model_name=EMBEDDING_MODEL_NAME,
    chunk_spec=chunk_spec
)

In [0]:
chunked_docs_table = compute_chunks(
    # The source documents table.
    docs_table=INPUT_DELTA_TABLE,
    # The column containing the documents to be chunked.
    doc_column="Plot",
    # The chunking function that takes a string (document) and returns a list of strings (chunks).
    chunk_fn=chunk_fn,
    # The output table with the chunked data
    chunked_docs_table=CHUNKED_DELTA_TABLE,
)

display(spark.read.table(chunked_docs_table))

In [0]:
spark.sql(f"ALTER TABLE {CHUNKED_DELTA_TABLE} SET TBLPROPERTIES (delta.enableChangeDataCapture = true)")


## Create the vector search index

In [0]:
from databricks.vector_search.client import VectorSearchClient


In [0]:
print(VECTOR_INDEX_NAME)

In [0]:
client = VectorSearchClient()

index = client.create_delta_sync_index(
  endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
  source_table_name=CHUNKED_DELTA_TABLE,
  index_name=VECTOR_INDEX_NAME,
  pipeline_type="TRIGGERED",
  primary_key="id",
  embedding_source_column="content_chunked",
  embedding_model_endpoint_name=EMBEDDING_MODEL_NAME
)

In [0]:
mlflow.end_run()